# Data Ingestion Part 2:
## Carbon Prices (ICAP)

### Capstone Project for the 2025 "Data Practictioner" Bootcamp" at neuefische
Author: Paul Ringler

This notebook documents the ETL pipeline for carbon prices in on the European Market (2015-2025)

Data is used with permission from ICAP at `https://icapcarbonaction.com/`
Further details on the data follows in `\capstone\notebooks\08_consolidated_dataset_EDA_and_FeatureEngineering.ipynb`

Objective: Ingest, clean and aggegrate data from the file provided by ICAP. Aggregate the data to daily, weekly and monthly resolution data because all other data is either daily or monthly and EDA and feature engineering will be done only on weekly and monthly data.

In [19]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from datetime import datetime
import warnings


The following code defines `setup_project_paths()` which gives file locations to the ETL pipeline, most critically the following:

    PROJECT_ROOT = Root path of the project folder
    RAW_DATA_PATH = Filepath of the raw data files
    PROCESSED_DATA_PATH = Filepath of the processed data files

In [20]:
# =============================================================================
# CELL 1: Configuration and Setup (Carbon Prices)
# =============================================================================
NOTEBOOK_NAME = "02: Carbon Prices Data Ingestion"
DATA_SUBFOLDER = "carbon_prices"
OBJECTIVE = "Process EU carbon prices (2015-2025)\nHandle structural break (<2019) vs (>2018)\nOutput: Daily, weekly, and monthly aggregated data\nMerge with electricity data"

# Carbon-specific configuration
CARBON_FILE_NAME = "icap-graph-price-data-2015-01-01-2025-09-12.csv"
DATE_COLUMN = "Date"
DELIMITER = ","  # Corrected from ";"
DECIMAL_SEPARATOR = "."

# Setup functions (reused from electricity - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")


NOTEBOOK: 02: Carbon Prices Data Ingestion
Start Time: 2025-09-29 13:57:09
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\carbon_prices
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process EU carbon prices (2015-2025)
Handle structural break (<2019) vs (>2018)
Output: Daily, weekly, and monthly aggregated data
Merge with electricity data
Cell 1: Configuration and Setup - READY


The pipeline starts with defining `standardize_missing_values()`, which functions identical to `01_electricity_data_ingestion.ipynb`.

In [21]:
# =============================================================================
# CELL 2: Missing Values Standardization Function 
# =============================================================================

def standardize_missing_values(df, additional_missing=None):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")


Cell 2: Missing Values Standardization Function - READY


Next, the pipeline discovers the data file to ingest, using `discover_carbon_file()`.

In [22]:
# =============================================================================
# CELL 3: Carbon File Discovery (Consistent Structure)
# =============================================================================

def discover_carbon_file(data_path, filename):
    """
    Discover single carbon prices file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary or None if not found
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of carbon file discovery."""
    print("Carbon file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with carbon data loading")
    else:
        print("ERROR: Carbon file not found!")

# Execute file discovery
carbon_file_info = discover_carbon_file(RAW_DATA_PATH, CARBON_FILE_NAME)
display_file_discovery_results(carbon_file_info)

print("Cell 3: Carbon File Discovery - COMPLETE")

Carbon file discovery:
  File: icap-graph-price-data-2015-01-01-2025-09-12.csv
  Exists: True
  Path: c:\Users\paulr\Desktop\Capstone\data\raw\carbon_prices\icap-graph-price-data-2015-01-01-2025-09-12.csv
Ready to proceed with carbon data loading
Cell 3: Carbon File Discovery - COMPLETE


In [23]:
# =============================================================================
# CELL 4: Load and Clean Carbon Data (Consistent with Electricity Metadata)
# =============================================================================

def load_complete_carbon_data(file_path):
    """
    Load complete carbon prices dataset with proper settings.
    
    Args:
        file_path (str): Path to carbon CSV file
    
    Returns:
        tuple: (dataframe, metadata) - consistent with electricity structure
    """
    try:
        # Load complete dataset
        df = pd.read_csv(file_path, 
                        delimiter=DELIMITER,
                        decimal=DECIMAL_SEPARATOR)
        
        # Clean missing values (consistent with electricity)
        df_clean, missing_patterns = standardize_missing_values(df)
        
        # Convert date column to datetime
        df_clean[DATE_COLUMN] = pd.to_datetime(df_clean[DATE_COLUMN], format='%Y-%m-%d')
        
        # Create metadata (consistent structure with electricity)
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'price_columns': [col for col in df_clean.columns if any(x in col for x in ['Primary Market', 'Secondary Market', 'Exchange rate'])],
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean[DATE_COLUMN].min(), df_clean[DATE_COLUMN].max()) if len(df_clean) > 0 else (None, None)
        }
        
        return df_clean, metadata
        
    except Exception as e:
        return None, {'error': str(e)}

# Load complete data
if carbon_file_info['exists']:
    carbon_df, carbon_metadata = load_complete_carbon_data(carbon_file_info['path'])
    
    if carbon_df is not None:
        print("COMPLETE CARBON DATA LOADED:")
        print("-" * 40)
        print(f"Shape: {carbon_metadata['rows']} rows x {len(carbon_metadata['columns'])} columns")
        print(f"Date range: {carbon_metadata['date_range'][0]} to {carbon_metadata['date_range'][1]}")
        print(f"Price columns: {carbon_metadata['price_columns']}")
        
        if carbon_metadata['missing_patterns_found']:
            print("Missing patterns found and standardized:")
            for col, pattern in carbon_metadata['missing_patterns_found'].items():
                print(f"  {col}: {pattern['converted_missing']} missing values converted")
        else:
            print("No text-based missing patterns found")
        
        print("\nSample data:")
        print(carbon_df.head(3))
    
    else:
        print(f"ERROR loading complete data: {carbon_metadata['error']}")

print("Cell 4: Load and Clean Carbon Data - COMPLETE")

COMPLETE CARBON DATA LOADED:
----------------------------------------
Shape: 2461 rows x 12 columns
Date range: 2015-01-02 00:00:00 to 2025-06-30 00:00:00
Price columns: ['Exchange rate EUR/EUR(<2019)', 'Exchange rate EUR/USD(<2019)', 'Primary Market(<2019)', 'Exchange rate EUR/EUR(>2018)', 'Exchange rate EUR/USD(>2018)', 'Primary Market(>2018)', 'Secondary Market(>2018)']
No text-based missing patterns found

Sample data:
        Date  Exchange rate EUR/EUR(<2019)  Exchange rate EUR/USD(<2019)  \
0 2015-01-02                           1.0                      1.162133   
1 2015-01-05                           1.0                      1.162133   
2 2015-01-06                           1.0                      1.162133   

  Market Currency(<2019)  Primary Market(<2019)     \
0                    EUR                    NaN      
1                    EUR                    NaN      
2                    EUR                    NaN      

   Exchange rate EUR/EUR(>2018)  Exchange rate EUR/

In [24]:
# =============================================================================
# CELL 5: Handle Structural Break (Temporal Consolidation)
# =============================================================================

def consolidate_carbon_structural_break(df):
    """
    Handle structural break through temporal consolidation, not renaming.
    
    Steps:
    1. Remove junk columns (empty or meaningless names)
    2. Create consolidated columns using temporal logic (before/after 2019-01-07)
    3. Drop original fragmented columns
    4. Drop unwanted columns (EUR/EUR, Market Currency)
    
    Args:
        df (DataFrame): Raw carbon data with structural break
    
    Returns:
        DataFrame: Data with properly consolidated columns
    """
    df_consolidated = df.copy()
    
    print("HANDLING STRUCTURAL BREAK (TEMPORAL CONSOLIDATION):")
    print("-" * 55)
    
    # Step 1: Remove junk columns (empty names, whitespace, meaningless identifiers)
    junk_columns = []
    for col in df_consolidated.columns:
        col_stripped = str(col).strip()
        if (col_stripped == '' or 
            col_stripped in [' ', '.1', '1', 'Unnamed'] or 
            col_stripped.startswith('Unnamed:') or
            df_consolidated[col].isna().all()):
            junk_columns.append(col)
    
    if junk_columns:
        print(f"Step 1: Removing junk columns: {junk_columns}")
        df_consolidated.drop(junk_columns, axis=1, inplace=True)
    else:
        print("Step 1: No junk columns found")
    
    # Step 2: Create consolidated columns using temporal logic
    print("\nStep 2: Creating temporally consolidated columns...")
    
    # Define the structural break date
    break_date = '2019-01-07'
    df_consolidated['temp_date'] = pd.to_datetime(df_consolidated[DATE_COLUMN])
    
    # Exchange rate EUR/USD consolidation
    eur_usd_before = 'Exchange rate EUR/USD(<2019)'
    eur_usd_after = 'Exchange rate EUR/USD(>2018)'
    
    if eur_usd_before in df_consolidated.columns and eur_usd_after in df_consolidated.columns:
        df_consolidated['Exchange rate EUR/USD'] = np.where(
            df_consolidated['temp_date'] < break_date,
            df_consolidated[eur_usd_before],
            df_consolidated[eur_usd_after]
        )
        
        before_count = df_consolidated[eur_usd_before].notna().sum()
        after_count = df_consolidated[eur_usd_after].notna().sum() 
        combined_count = df_consolidated['Exchange rate EUR/USD'].notna().sum()
        
        print(f"  Exchange rate EUR/USD:")
        print(f"    Before {break_date}: {before_count} values")
        print(f"    After {break_date}: {after_count} values")
        print(f"    Combined: {combined_count} values")
        
        # Drop original columns
        df_consolidated.drop([eur_usd_before, eur_usd_after], axis=1, inplace=True)
    else:
        print(f"  WARNING: Could not find EUR/USD columns")
    
    # Primary Market consolidation  
    primary_before = 'Primary Market(<2019)'
    primary_after = 'Primary Market(>2018)'
    
    if primary_before in df_consolidated.columns and primary_after in df_consolidated.columns:
        df_consolidated['Primary Market'] = np.where(
            df_consolidated['temp_date'] < break_date,
            df_consolidated[primary_before],
            df_consolidated[primary_after]
        )
        
        before_count = df_consolidated[primary_before].notna().sum()
        after_count = df_consolidated[primary_after].notna().sum()
        combined_count = df_consolidated['Primary Market'].notna().sum()
        
        print(f"  Primary Market:")
        print(f"    Before {break_date}: {before_count} values")
        print(f"    After {break_date}: {after_count} values")
        print(f"    Combined: {combined_count} values")
        
        # Drop original columns
        df_consolidated.drop([primary_before, primary_after], axis=1, inplace=True)
    else:
        print(f"  WARNING: Could not find Primary Market columns")
    
    # Secondary Market (only exists after break)
    secondary_after = 'Secondary Market(>2018)'
    
    if secondary_after in df_consolidated.columns:
        # Simply rename - no consolidation needed (only exists after break)
        df_consolidated['Secondary Market'] = df_consolidated[secondary_after]
        after_count = df_consolidated['Secondary Market'].notna().sum()
        
        print(f"  Secondary Market:")
        print(f"    Only after {break_date}: {after_count} values")
        
        # Drop original column
        df_consolidated.drop([secondary_after], axis=1, inplace=True)
    else:
        print(f"  WARNING: Could not find Secondary Market column")
    
    # Step 3: Drop unwanted columns (EUR/EUR always 1.0, Market Currency always EUR)
    print("\nStep 3: Removing unwanted columns...")
    columns_to_drop = []
    for col in df_consolidated.columns:
        if ('Exchange rate EUR/EUR' in str(col) or 
            'Market Currency' in str(col) or
            col == 'temp_date'):
            columns_to_drop.append(col)
    
    if columns_to_drop:
        print(f"  Dropping: {columns_to_drop}")
        df_consolidated.drop(columns_to_drop, axis=1, inplace=True)
    else:
        print("  No unwanted columns to drop")
    
    # Final verification
    print(f"\nFINAL CONSOLIDATION RESULTS:")
    print(f"  Final columns: {list(df_consolidated.columns)}")
    
    # Verify data distribution across time periods
    df_consolidated['temp_date'] = pd.to_datetime(df_consolidated[DATE_COLUMN])
    before_break = df_consolidated[df_consolidated['temp_date'] < break_date]
    after_break = df_consolidated[df_consolidated['temp_date'] >= break_date]
    
    print(f"\nDATA VERIFICATION:")
    print(f"  Total rows: {len(df_consolidated)}")
    print(f"  Before {break_date}: {len(before_break)} rows")
    print(f"  After {break_date}: {len(after_break)} rows")
    
    # Show non-null counts by period
    for col in df_consolidated.columns:
        if col not in [DATE_COLUMN, 'temp_date']:
            before_count = before_break[col].notna().sum()
            after_count = after_break[col].notna().sum()
            print(f"  {col}: {before_count} before + {after_count} after = {before_count + after_count} total")
    
    # Clean up temporary column
    df_consolidated.drop('temp_date', axis=1, inplace=True)
    
    return df_consolidated

# Execute structural break consolidation
if carbon_df is not None:
    carbon_consolidated = consolidate_carbon_structural_break(carbon_df)
    
    print("\nSample of consolidated data:")
    print(carbon_consolidated.head(5))
    
    print("\nData around structural break:")
    # Show data around 2019-01-07 break point
    break_sample = carbon_consolidated[
        (pd.to_datetime(carbon_consolidated['Date']) >= '2018-12-01') & 
        (pd.to_datetime(carbon_consolidated['Date']) <= '2019-02-01')
    ]
    
    if len(break_sample) > 0:
        print(break_sample[['Date', 'Exchange rate EUR/USD', 'Primary Market', 'Secondary Market']].head(10))
    
    print("\nRecent data sample (should show Secondary Market values):")
    recent_sample = carbon_consolidated[pd.to_datetime(carbon_consolidated['Date']) >= '2020-01-01'].head(5)
    print(recent_sample[['Date', 'Exchange rate EUR/USD', 'Primary Market', 'Secondary Market']])

print("Cell 5: Handle Structural Break (Complete Rewrite) - COMPLETE")

HANDLING STRUCTURAL BREAK (TEMPORAL CONSOLIDATION):
-------------------------------------------------------
Step 1: Removing junk columns: [' ', ' .1']

Step 2: Creating temporally consolidated columns...
  Exchange rate EUR/USD:
    Before 2019-01-07: 1018 values
    After 2019-01-07: 1443 values
    Combined: 2461 values
  Primary Market:
    Before 2019-01-07: 804 values
    After 2019-01-07: 1409 values
    Combined: 2213 values
  Secondary Market:
    Only after 2019-01-07: 306 values

Step 3: Removing unwanted columns...
  Dropping: ['Exchange rate EUR/EUR(<2019)', 'Market Currency(<2019)', 'Exchange rate EUR/EUR(>2018)', 'Market Currency(>2018)', 'temp_date']

FINAL CONSOLIDATION RESULTS:
  Final columns: ['Date', 'Exchange rate EUR/USD', 'Primary Market', 'Secondary Market']

DATA VERIFICATION:
  Total rows: 2461
  Before 2019-01-07: 1018 rows
  After 2019-01-07: 1443 rows
  Exchange rate EUR/USD: 1018 before + 1443 after = 2461 total
  Primary Market: 804 before + 1409 after =

In [25]:
# =============================================================================
# CELL 6: Standardize Column Names
# =============================================================================

def standardize_carbon_column_names(df):
    """
    Standardize column names with carbonprices_ prefix and clean formatting.
    
    Args:
        df (DataFrame): Data with consolidated columns
    
    Returns:
        DataFrame: Data with standardized column names
    """
    df_renamed = df.copy()
    
    # Define column renaming map
    rename_map = {}
    
    for col in df.columns:
        if col == DATE_COLUMN:
            # Keep date column as is
            continue
        else:
            # Create standardized name
            new_name = col.lower()
            new_name = new_name.replace(' ', '_')
            new_name = new_name.replace('/', '_')
            new_name = new_name.replace('-', '_')
            new_name = new_name.replace('(', '').replace(')', '')
            new_name = new_name.replace('>', '')  # Remove > characters
            new_name = f"carbonprices_{new_name}"
            rename_map[col] = new_name
    
    # Apply renaming
    df_renamed.rename(columns=rename_map, inplace=True)
    
    print("COLUMN NAME STANDARDIZATION:")
    print("-" * 35)
    for old_name, new_name in rename_map.items():
        print(f"  {old_name} → {new_name}")
    
    print(f"\nFINAL STANDARDIZED COLUMNS:")
    print(f"  {list(df_renamed.columns)}")
    
    return df_renamed

# Execute column standardization
if 'carbon_consolidated' in locals():
    carbon_standardized = standardize_carbon_column_names(carbon_consolidated)
    
    print("\nSample data with new names:")
    print(carbon_standardized.head(3))

print("Cell 6: Standardize Column Names - COMPLETE")

COLUMN NAME STANDARDIZATION:
-----------------------------------
  Exchange rate EUR/USD → carbonprices_exchange_rate_eur_usd
  Primary Market → carbonprices_primary_market
  Secondary Market → carbonprices_secondary_market

FINAL STANDARDIZED COLUMNS:
  ['Date', 'carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market']

Sample data with new names:
        Date  carbonprices_exchange_rate_eur_usd  carbonprices_primary_market  \
0 2015-01-02                            1.162133                          NaN   
1 2015-01-05                            1.162133                          NaN   
2 2015-01-06                            1.162133                          NaN   

   carbonprices_secondary_market  
0                            NaN  
1                            NaN  
2                            NaN  
Cell 6: Standardize Column Names - COMPLETE


In [26]:
# =============================================================================
# CELL 7: Create Aggregations (Daily/Weekly/Monthly - Corrected)
# =============================================================================

def create_carbon_aggregations(df, date_col='Date'):
    """
    Create daily, weekly, and monthly aggregations for carbon data.
    Different handling for price columns (Int64) vs exchange rates (float64).
    
    Args:
        df (DataFrame): Carbon data with standardized columns
        date_col (str): Name of date column
    
    Returns:
        dict: Dictionary with 'daily', 'weekly', 'monthly' DataFrames
    """
    if df is None or len(df) == 0:
        return {'daily': None, 'weekly': None, 'monthly': None}
    
    # Ensure date column is datetime and set as index
    df_work = df.copy()
    df_work[date_col] = pd.to_datetime(df_work[date_col])
    df_indexed = df_work.set_index(date_col)
    
    # Get carbon columns and categorize by data type treatment
    carbon_columns = [col for col in df_indexed.columns if col.startswith('carbonprices_')]
    
    # Separate price columns (to be rounded) from exchange rate columns (keep float)
    price_columns = [col for col in carbon_columns if 'market' in col.lower()]
    exchange_rate_columns = [col for col in carbon_columns if 'exchange_rate' in col.lower()]
    
    print(f"COLUMN CATEGORIZATION:")
    print(f"  Price columns (will be Int64): {price_columns}")
    print(f"  Exchange rate columns (will stay float64): {exchange_rate_columns}")
    
    # Convert all carbon columns to numeric UPFRONT
    print(f"\nConverting {len(carbon_columns)} columns to numeric...")
    for col in carbon_columns:
        df_indexed[col] = pd.to_numeric(df_indexed[col], errors='coerce')
        non_null_count = df_indexed[col].notna().sum()
        print(f"  {col}: {non_null_count} valid numeric values")
    
    aggregations = {}
    
    # Daily aggregation (data is already daily, just format)
    daily = df_indexed.copy()
    daily['date'] = daily.index.date
    daily['aggregation_level'] = 'daily'
    
    # Apply different data type handling
    for col in price_columns:
        daily[col] = daily[col].round().astype('Int64')
    
    for col in exchange_rate_columns:
        daily[col] = daily[col].astype('float64')  # Keep as float64
    
    aggregations['daily'] = daily.reset_index(drop=True)
    
    # Weekly aggregation (ISO weeks, Monday start)
    weekly = df_indexed[carbon_columns].resample('W-MON').mean()
    weekly['date'] = weekly.index.date
    weekly['aggregation_level'] = 'weekly'
    
    # Apply different data type handling
    for col in price_columns:
        weekly[col] = weekly[col].round().astype('Int64')
    
    for col in exchange_rate_columns:
        weekly[col] = weekly[col].astype('float64')
    
    aggregations['weekly'] = weekly.reset_index(drop=True)
    
    # Monthly aggregation (first of month)
    monthly = df_indexed[carbon_columns].resample('MS').mean()
    monthly['date'] = monthly.index.date
    monthly['aggregation_level'] = 'monthly'
    
    # Apply different data type handling
    for col in price_columns:
        monthly[col] = monthly[col].round().astype('Int64')
    
    for col in exchange_rate_columns:
        monthly[col] = monthly[col].astype('float64')
    
    aggregations['monthly'] = monthly.reset_index(drop=True)
    
    print("\nCARBON AGGREGATION RESULTS:")
    print("-" * 30)
    for level, agg_df in aggregations.items():
        if agg_df is not None:
            print(f"  {level.capitalize()}: {len(agg_df)} periods")
            print(f"    Date range: {agg_df['date'].min()} to {agg_df['date'].max()}")
            
            # Show data types
            carbon_cols = [col for col in agg_df.columns if col.startswith('carbonprices_')]
            for col in carbon_cols:
                dtype = agg_df[col].dtype
                print(f"    {col}: {dtype}")
    
    return aggregations

# Execute aggregations
if 'carbon_standardized' in locals():
    carbon_aggregations = create_carbon_aggregations(carbon_standardized)
    
    # Show sample from each aggregation level with data types
    for level in ['daily', 'weekly', 'monthly']:
        agg_df = carbon_aggregations[level]
        if agg_df is not None:
            print(f"\nSample {level} data:")
            print(agg_df.head(3))

print("Cell 7: Create Aggregations - COMPLETE")

COLUMN CATEGORIZATION:
  Price columns (will be Int64): ['carbonprices_primary_market', 'carbonprices_secondary_market']
  Exchange rate columns (will stay float64): ['carbonprices_exchange_rate_eur_usd']

Converting 3 columns to numeric...
  carbonprices_exchange_rate_eur_usd: 2461 valid numeric values
  carbonprices_primary_market: 2213 valid numeric values
  carbonprices_secondary_market: 306 valid numeric values

CARBON AGGREGATION RESULTS:
------------------------------
  Daily: 2461 periods
    Date range: 2015-01-02 to 2025-06-30
    carbonprices_exchange_rate_eur_usd: float64
    carbonprices_primary_market: Int64
    carbonprices_secondary_market: Int64
  Weekly: 548 periods
    Date range: 2015-01-05 to 2025-06-30
    carbonprices_exchange_rate_eur_usd: float64
    carbonprices_primary_market: Int64
    carbonprices_secondary_market: Int64
  Monthly: 126 periods
    Date range: 2015-01-01 to 2025-06-01
    carbonprices_exchange_rate_eur_usd: float64
    carbonprices_primary_m

In [27]:
# =============================================================================
# CELL 8: Transform to Long Format (with data_completeness)
# =============================================================================

def transform_carbon_to_long_format(aggregations_dict):
    """
    Transform carbon aggregations to long format matching electricity structure.
    
    Args:
        aggregations_dict (dict): Dictionary of aggregated carbon DataFrames
    
    Returns:
        DataFrame: Carbon data in long format
    """
    all_data = []
    
    for level in ['daily', 'weekly', 'monthly']:
        df = aggregations_dict[level]
        if df is None or len(df) == 0:
            continue
        
        # Convert date to datetime for calculations
        df['date'] = pd.to_datetime(df['date'])
        
        # Add time components (consistent with electricity)
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week'] = df['date'].dt.isocalendar().week
        df['month_name'] = df['date'].dt.month_name()
        
        # Calculate data_completeness for carbon data
        # For carbon: daily data expected, so completeness = (non-null values / 1) * 100
        carbon_columns = [col for col in df.columns if col.startswith('carbonprices_')]
        
        if carbon_columns:
            # Count non-null values across all carbon price columns
            non_null_counts = df[carbon_columns].notna().sum(axis=1)
            max_possible = len(carbon_columns)  # All carbon columns should have data
            df['data_completeness'] = (non_null_counts / max_possible * 100).round(1)
        else:
            df['data_completeness'] = pd.Series(0.0, index=df.index)
        
        # Convert date back to string in ISO format
        df['date'] = df['date'].dt.strftime('%Y-%m-%d')
        
        # Select columns in correct order (consistent with electricity)
        base_columns = [
            'date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name'
        ]
        
        carbon_columns = [col for col in df.columns if col.startswith('carbonprices_')]
        other_columns = ['data_completeness']
        
        final_columns = base_columns + carbon_columns + other_columns
        existing_columns = [col for col in final_columns if col in df.columns]
        
        df_final = df[existing_columns].copy()
        all_data.append(df_final)
    
    # Concatenate all data
    if all_data:
        result = pd.concat(all_data, ignore_index=True)
        result = result.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        return result
    else:
        return pd.DataFrame()

# Execute long format transformation
if 'carbon_aggregations' in locals():
    carbon_long_format = transform_carbon_to_long_format(carbon_aggregations)
    
    print("CARBON LONG FORMAT RESULTS:")
    print("-" * 30)
    print(f"  Total rows: {len(carbon_long_format)}")
    print(f"  Columns: {carbon_long_format.columns.tolist()}")
    print(f"  Shape: {carbon_long_format.shape}")
    
    if len(carbon_long_format) > 0:
        print(f"  Date range: {carbon_long_format['date'].min()} to {carbon_long_format['date'].max()}")
        print(f"  Aggregation levels: {carbon_long_format['aggregation_level'].value_counts().to_dict()}")
        
        print("\nSample of final long format data:")
        print(carbon_long_format.head())

print("Cell 8: Transform to Long Format - COMPLETE")

CARBON LONG FORMAT RESULTS:
------------------------------
  Total rows: 3135
  Columns: ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name', 'carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market', 'data_completeness']
  Shape: (3135, 11)
  Date range: 2015-01-01 to 2025-06-30
  Aggregation levels: {'daily': 2461, 'weekly': 548, 'monthly': 126}

Sample of final long format data:
         date  year  month  quarter  week aggregation_level month_name  \
0  2015-01-01  2015      1        1     1           monthly    January   
1  2015-01-02  2015      1        1     1             daily    January   
2  2015-01-05  2015      1        1     2             daily    January   
3  2015-01-05  2015      1        1     2            weekly    January   
4  2015-01-06  2015      1        1     2             daily    January   

   carbonprices_exchange_rate_eur_usd  carbonprices_primary_market  \
0                            1

In [28]:
# =============================================================================
# CELL 9: Save Carbon Dataset (Validation Sample)
# =============================================================================

def save_carbon_dataset(df, output_dir, filename="carbon_consolidated.csv"):
    """
    Save carbon dataset as separate validation sample.
    
    Args:
        df (DataFrame): Final carbon dataset
        output_dir (Path): Directory for outputs
        filename (str): Output filename
    
    Returns:
        Path: Path to saved file
    """
    if df is None or len(df) == 0:
        print("No carbon data to save!")
        return None
    
    output_path = output_dir / filename
    df.to_csv(output_path, index=False)
    
    print(f"CARBON DATASET SAVED:")
    print(f"  Path: {output_path}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    
    return output_path

# Save carbon dataset as validation sample
if 'carbon_long_format' in locals():
    carbon_file_path = save_carbon_dataset(carbon_long_format, PROCESSED_DATA_PATH)

print("Cell 9: Save Carbon Dataset - COMPLETE")

CARBON DATASET SAVED:
  Path: c:\Users\paulr\Desktop\Capstone\data\processed\carbon_consolidated.csv
  Rows: 3135
  Columns: 11
Cell 9: Save Carbon Dataset - COMPLETE


In [29]:
# =============================================================================
# CELL 10: Merge with Electricity Data
# =============================================================================

def merge_carbon_with_electricity(carbon_df, electricity_file_path, output_dir):
    """
    Merge carbon data with existing electricity data.
    
    Args:
        carbon_df (DataFrame): Carbon dataset in long format
        electricity_file_path (str/Path): Path to electricity_consolidated.csv
        output_dir (Path): Directory for final output
    
    Returns:
        DataFrame: Merged dataset
    """
    electricity_path = Path(electricity_file_path)
    
    # Load electricity data
    if not electricity_path.exists():
        print(f"ERROR: Electricity file not found at {electricity_path}")
        return pd.DataFrame()
    
    print("MERGING CARBON WITH ELECTRICITY:")
    print("-" * 40)
    
    try:
        electricity_df = pd.read_csv(electricity_path)
        print(f"  Electricity data loaded: {electricity_df.shape}")
        print(f"  Carbon data: {carbon_df.shape}")
        
        # Merge on date and aggregation_level
        merged_df = pd.merge(
            electricity_df, 
            carbon_df, 
            on=['date', 'aggregation_level'], 
            how='outer',  # Keep all dates from both datasets
            suffixes=('', '_carbon')  # Handle duplicate columns
        )
        
        print(f"  Merged data: {merged_df.shape}")
        
        # Handle duplicate columns (keep original structure, add carbon columns)
        duplicate_cols = ['year', 'month', 'quarter', 'week', 'month_name', 'data_completeness']
        for col in duplicate_cols:
            carbon_col = f"{col}_carbon"
            if carbon_col in merged_df.columns:
                # Use electricity values where available, fill with carbon values
                merged_df[col] = merged_df[col].fillna(merged_df[carbon_col])
                merged_df.drop(carbon_col, axis=1, inplace=True)
        
        # Sort by date and aggregation level
        merged_df = merged_df.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Save merged dataset as data_consolidated.csv
        final_path = output_dir / "data_consolidated.csv"
        merged_df.to_csv(final_path, index=False)
        
        print(f"\nFINAL CONSOLIDATED DATASET:")
        print(f"  Saved to: {final_path}")
        print(f"  Final shape: {merged_df.shape}")
        print(f"  Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")
        
        return merged_df
        
    except Exception as e:
        print(f"ERROR during merge: {e}")
        return pd.DataFrame()

# Execute merge with electricity data
if 'carbon_long_format' in locals():
    electricity_file = PROCESSED_DATA_PATH / "electricity_consolidated.csv"
    final_merged_df = merge_carbon_with_electricity(carbon_long_format, electricity_file, PROCESSED_DATA_PATH)
    
    if len(final_merged_df) > 0:
        print("\nSample of merged data:")
        print(final_merged_df.head())
        print("\nColumn overview:")
        print(final_merged_df.columns.tolist())

print("Cell 10: Merge with Electricity Data - COMPLETE")


MERGING CARBON WITH ELECTRICITY:
----------------------------------------
  Electricity data loaded: (4590, 12)
  Carbon data: (3135, 11)
  Merged data: (4590, 21)

FINAL CONSOLIDATED DATASET:
  Saved to: c:\Users\paulr\Desktop\Capstone\data\processed\data_consolidated.csv
  Final shape: (4590, 15)
  Date range: 2015-01-01 to 2025-09-01

Sample of merged data:
         date  year  month  quarter  week aggregation_level  price_exaa_mean  \
0  2015-01-01  2015      1        1     1             daily             44.0   
1  2015-01-01  2015      1        1     1           monthly             30.0   
2  2015-01-02  2015      1        1     1             daily             31.0   
3  2015-01-03  2015      1        1     1             daily             19.0   
4  2015-01-04  2015      1        1     1             daily             14.0   

   price_mc_auction_mean  price_count_exaa  price_count_mc  data_completeness  \
0                    NaN                96               0              100

In [31]:
# =============================================================================
# CELL 11: Master Carbon Orchestration
# =============================================================================

def consolidate_carbon_data_complete(file_info, output_dir):
    """
    Master function to orchestrate complete carbon data consolidation and merge.
    
    Args:
        file_info (dict): Carbon file information dictionary
        output_dir (Path): Directory for outputs
    
    Returns:
        DataFrame: Final consolidated dataset (electricity + carbon)
    """
    print("="*60)
    print("CARBON DATA CONSOLIDATION - MASTER PROCESS")
    print("="*60)
    
    if not file_info['exists']:
        print("ERROR: Carbon file not found!")
        return pd.DataFrame()
    
    # Step 1: Load data
    print("\nSTEP 1: Loading and cleaning carbon file...")
    df, metadata = load_complete_carbon_data(file_info['path'])
    
    if df is None:
        print(f"ERROR: {metadata['error']}")
        return pd.DataFrame()
    
    print(f"  SUCCESS: {metadata['rows']} rows loaded")
    print(f"  Missing patterns processed: {len(metadata['missing_patterns_found'])}")
    
    # Step 2: Handle structural break
    print("\nSTEP 2: Handling structural break...")
    df_consolidated = consolidate_carbon_structural_break(df)
    print(f"  SUCCESS: {len(df_consolidated.columns)} columns after consolidation")
    
    # Step 3: Standardize names
    print("\nSTEP 3: Standardizing column names...")
    df_standardized = standardize_carbon_column_names(df_consolidated)
    print(f"  SUCCESS: Standardized to {len(df_standardized.columns)} columns")
    
    # Step 4: Create aggregations
    print("\nSTEP 4: Creating aggregations...")
    aggregations = create_carbon_aggregations(df_standardized)
    
    for level, agg_df in aggregations.items():
        if agg_df is not None:
            print(f"  {level}: {len(agg_df)} periods")
    
    # Step 5: Transform to long format
    print("\nSTEP 5: Transforming to long format...")
    carbon_df = transform_carbon_to_long_format(aggregations)
    
    if len(carbon_df) > 0:
        print(f"  SUCCESS: {len(carbon_df)} total rows in carbon dataset")
        print(f"  Date range: {carbon_df['date'].min()} to {carbon_df['date'].max()}")
        print(f"  Shape: {carbon_df.shape}")
    else:
        print("  ERROR: No data in carbon dataset")
        return pd.DataFrame()
    
    # Step 6: Save carbon dataset (validation sample)
    print("\nSTEP 6: Saving carbon dataset...")
    save_carbon_dataset(carbon_df, output_dir)
    
    # Step 7: Merge with electricity
    print("\nSTEP 7: Merging with electricity data...")
    electricity_path = output_dir / "electricity_consolidated.csv"
    final_df = merge_carbon_with_electricity(carbon_df, electricity_path, output_dir)
    
    if len(final_df) > 0:
        print(f"\nFINAL SUCCESS:")
        print(f"  Carbon dataset: carbon_consolidated.csv")
        print(f"  Final merged: data_consolidated.csv")
        print(f"  Total rows: {len(final_df)}")
        print(f"  Total columns: {len(final_df.columns)}")
        
        return final_df
    else:
        print("ERROR: Merge failed")
        return pd.DataFrame()

print("Cell 11: Master Carbon Orchestration Function - READY")
print("\n" + "="*60)
print("ALL CORRECTED CARBON FUNCTIONS READY FOR TESTING")
print("Use: final_df = consolidate_carbon_data_complete(carbon_file_info, PROCESSED_DATA_PATH)")
print("="*60)

Cell 11: Master Carbon Orchestration Function - READY

ALL CORRECTED CARBON FUNCTIONS READY FOR TESTING
Use: final_df = consolidate_carbon_data_complete(carbon_file_info, PROCESSED_DATA_PATH)


In [32]:
final_df = consolidate_carbon_data_complete(carbon_file_info, PROCESSED_DATA_PATH)

CARBON DATA CONSOLIDATION - MASTER PROCESS

STEP 1: Loading and cleaning carbon file...
  SUCCESS: 2461 rows loaded
  Missing patterns processed: 0

STEP 2: Handling structural break...
HANDLING STRUCTURAL BREAK (TEMPORAL CONSOLIDATION):
-------------------------------------------------------
Step 1: Removing junk columns: [' ', ' .1']

Step 2: Creating temporally consolidated columns...
  Exchange rate EUR/USD:
    Before 2019-01-07: 1018 values
    After 2019-01-07: 1443 values
    Combined: 2461 values
  Primary Market:
    Before 2019-01-07: 804 values
    After 2019-01-07: 1409 values
    Combined: 2213 values
  Secondary Market:
    Only after 2019-01-07: 306 values

Step 3: Removing unwanted columns...
  Dropping: ['Exchange rate EUR/EUR(<2019)', 'Market Currency(<2019)', 'Exchange rate EUR/EUR(>2018)', 'Market Currency(>2018)', 'temp_date']

FINAL CONSOLIDATION RESULTS:
  Final columns: ['Date', 'Exchange rate EUR/USD', 'Primary Market', 'Secondary Market']

DATA VERIFICATION:


In [33]:
# Carbon dataset overview
for col in final_df.columns:
    dtype = final_df[col].dtype
    missing = final_df[col].isnull().sum()
    unique = final_df[col].nunique()
    
    if dtype in ['float64', 'int64', 'Int64']:
        min_val = final_df[col].min()
        max_val = final_df[col].max()
        print(f"{col}: {dtype} | Missing: {missing} | Unique: {unique} | Range: {min_val} to {max_val}")
    else:
        print(f"{col}: {dtype} | Missing: {missing} | Unique: {unique}")

date: object | Missing: 0 | Unique: 3897
year: int64 | Missing: 0 | Unique: 11 | Range: 2015 to 2025
month: int64 | Missing: 0 | Unique: 12 | Range: 1 to 12
quarter: int64 | Missing: 0 | Unique: 4 | Range: 1 to 4
week: int64 | Missing: 0 | Unique: 53 | Range: 1 to 53
aggregation_level: object | Missing: 0 | Unique: 3
price_exaa_mean: float64 | Missing: 1150 | Unique: 383 | Range: -13.0 to 727.0
price_mc_auction_mean: float64 | Missing: 2151 | Unique: 382 | Range: -23.0 to 764.0
price_count_exaa: int64 | Missing: 0 | Unique: 21 | Range: 0 to 2976
price_count_mc: int64 | Missing: 0 | Unique: 15 | Range: 0 to 2976
data_completeness: float64 | Missing: 0 | Unique: 30 | Range: 0.0 to 200.0
month_name: object | Missing: 0 | Unique: 12
carbonprices_exchange_rate_eur_usd: float64 | Missing: 1454 | Unique: 205 | Range: 0.982566667 to 1.23478
carbonprices_primary_market: Int64 | Missing: 1731 | Unique: 92 | Range: 4 to 98
carbonprices_secondary_market: Int64 | Missing: 3901 | Unique: 74 | Range: